In [1]:
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict, GridSearchCV
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pygad
from sklearn.metrics import mean_squared_error as mse
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sys import stdout
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer
import time
from sklearn.svm import SVR
df = pd.read_excel("datasets/Dataset_2.xlsx",sheet_name="preproc")
import tensorflow as tf
from keras.callbacks import EarlyStopping
import gc
import psutil
from multiprocessing import Pool, cpu_count, get_context

trials = 10
random_cv = np.random.seed(777)
seeds = np.random.choice(trials + 1, size=trials, replace= False)
n_components = 60

print(tf.config.list_physical_devices('GPU'))



KeyboardInterrupt: 

In [ ]:
import tensorflow as tf

# List available devices
print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

# Simple GPU test
with tf.device('/GPU:0'):
    a = tf.random.normal([1000, 1000])
    b = tf.random.normal([1000, 1000])
    c = tf.matmul(a, b)

print("Matrix multiplication complete!")


TensorFlow version: 2.18.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Matrix multiplication complete!


INPUT DATA

In [ ]:
vip_bands = []
with open(f"VIPs_350_Fixed_25_Bands.txt","r") as f:
    vip_bands = f.read().splitlines()

vip_bands = [float(vip_band) for vip_band in vip_bands]
print(vip_bands)

[379.5, 383.5, 387.5, 391.5, 395.5, 399.5, 403.5, 407.5, 411.5, 415.5, 419.5, 423.5, 427.5, 431.5, 435.5, 439.5, 443.5, 447.5, 451.5, 455.5, 459.5, 463.5, 467.5, 471.5, 475.5, 479.5, 483.5, 487.5, 491.5, 495.5, 499.5, 503.5, 507.5, 511.5, 515.5, 519.5, 523.5, 527.5, 531.5, 535.5, 539.5, 543.5, 547.5, 551.5, 555.5, 559.5, 563.5, 567.5, 571.5, 575.5, 579.5, 583.5, 587.5, 591.5, 595.5, 599.5, 603.5, 607.5, 611.5, 615.5, 619.5, 623.5, 627.5, 631.5, 635.5, 639.5, 643.5, 647.5, 651.5, 655.5, 659.5, 663.5, 667.5, 671.5, 675.5, 679.5, 683.5, 687.5, 691.5, 695.5, 699.5, 703.5, 707.5, 711.5, 715.5, 719.5, 723.5, 727.5, 731.5, 735.5, 739.5, 743.5, 747.5, 751.5, 755.5, 759.5, 763.5, 767.5, 771.5, 775.5, 779.5, 783.5, 787.5, 791.5, 795.5, 799.5, 803.5, 807.5, 811.5, 815.5, 819.5, 823.5, 827.5, 831.5, 835.5, 907.5, 911.5, 915.5, 919.5, 923.5, 927.5, 931.5, 935.5, 939.5, 943.5, 947.5, 951.5, 955.5, 959.5, 963.5, 967.5, 971.5, 975.5, 979.5, 983.5, 987.5, 991.5, 1015.5, 1019.5, 1023.5, 1027.5, 1031.5, 

In [ ]:
X = (df.loc[:,vip_bands]).values
print(X.shape)


(2905, 350)


OUTPUT DATA

In [ ]:
y = (df.iloc[:,7]).values
print(y.shape)

(2905,)


**NEGNRMSEPiqr**

In [ ]:


def NEGNRMSEPiqr(observed_values, predicted_values):
    # Calculate RMSEP
    rmsep = np.sqrt(np.mean((observed_values - predicted_values) ** 2))
    # Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = np.percentile(observed_values, 25)
    Q3 = np.percentile(observed_values, 75)

    # Calculate IQR
    IQR = Q3 - Q1

    return -rmsep/IQR


NEGNRMSEPiqrscorer = make_scorer(NEGNRMSEPiqr)

**NRMSEPiqr**

In [ ]:

def NRMSEPiqr(observed_values, predicted_values):
    # Calculate RMSEP
    rmsep = np.sqrt(np.mean((observed_values - predicted_values) ** 2))
    # Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = np.percentile(observed_values, 25)
    Q3 = np.percentile(observed_values, 75)

    # Calculate IQR
    IQR = Q3 - Q1

    return rmsep/IQR


NRMSEPiqrscorer = make_scorer(NRMSEPiqr)

Fitness Function

In [ ]:
def build_nn(input_size, filters=32, kernel_size=3, dropout_rate=0.2, learning_rate=0.001):

        '''
        inputs = tf.keras.Input(shape=(self._m,))
        x = tf.keras.layers.Reshape((self._m, 1))(inputs)
        x = tf.keras.layers.Conv1D(filters=filters, kernel_size=kernel_size, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
        x = tf.keras.layers.Conv1D(filters=filters, kernel_size=kernel_size, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.GlobalAveragePooling1D()(x)
        outputs = tf.keras.layers.Dense(1, activation='linear')(x)

        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mean_squared_error')
        return model
        '''
        model = tf.keras.models.Sequential()
        model.add(tf.keras.layers.Input(shape=(input_size,)))
        model.add(tf.keras.layers.Reshape((input_size, 1)))
        # Convolutional layers
        model.add(tf.keras.layers.Conv1D(filters=32, kernel_size=kernel_size, padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.06), activation='elu'))
        model.add(tf.keras.layers.Conv1D(filters=64, kernel_size=kernel_size, padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.06), activation='elu'))
        # Fully connected layers
        model.add(tf.keras.layers.Flatten())
        model.add(tf.keras.layers.Dense(64, activation='elu'))
        model.add(tf.keras.layers.Dropout(0.2))
        model.add(tf.keras.layers.Dense(1, activation='linear'))
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mean_squared_error')
        return model



def train_and_evaluate_nn(args):
        X_train, X_test, Y_train, Y_test = args
        model = build_nn(input_size=X_train.shape[1])   

        early_stopping = EarlyStopping(
            monitor='val_loss',
            min_delta=1e-4,       
            mode='auto',         
            patience=50,
            restore_best_weights=True, 
        )
        
        model.fit(X_train, Y_train, epochs=500, batch_size=32, verbose=0, validation_split=0.2, callbacks=[early_stopping])
        Y_pred = model.predict(X_test, verbose=0).flatten()
        fitness = NEGNRMSEPiqr(Y_test, Y_pred)
        tf.keras.backend.clear_session()
        gc.collect()
        return fitness





def fitness_function(ga_instance, solution, solution_idx):

    epsilon=0.3178
    C = 5164
    alpha = 0.00122457013067159
    NNRMSE = []
    # Select wavelengths based on the solution
    selected_wavelengths = np.where(solution == 1)[0]
    
    #if len(selected_wavelengths) >125:
    #    return -1e6
    X_selected = X[:, selected_wavelengths]

    data_list = []
    for i in range(trials):
        X_train, X_test, Y_train, Y_test = train_test_split(X_selected, y, test_size=0.25, random_state=seeds[i])
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model = build_nn(input_size=X_train_scaled .shape[1])   

        early_stopping = EarlyStopping(
            monitor='val_loss',
            min_delta=1e-4,       
            mode='auto',         
            patience=50,
            restore_best_weights=True, 
        )
        
        model.fit(X_train_scaled, Y_train, epochs=500, batch_size=32, verbose=0, validation_split=0.2, callbacks=[early_stopping])
        Y_pred = model.predict(X_test_scaled, verbose=0).flatten()
        NNRMSE.append(NEGNRMSEPiqr(Y_test, Y_pred))

    
    # Use the mean score as the fitness value (higher is better)
    return np.mean(NNRMSE)


Main

In [ ]:
# Number of wavelengths (genes)
num_wavelengths = X.shape[1]

# Create the GA instance
ga_instance = pygad.GA(
    num_generations=50,             # Number of generations
    num_parents_mating=1,          # Number of parents for crossover
    keep_elitism=1,
    fitness_func=fitness_function,  # Fitness function
    sol_per_pop=2,                 # Population size
    num_genes=num_wavelengths,      # Number of wavelengths
    gene_type=int,                  # Binary genes (0 or 1)
    init_range_low=0,               # Genes initialized to 0
    init_range_high=2,              # Genes initialized to 1 (binary)
    parent_selection_type="sss",    # Stochasting sampling selection
    crossover_type="single_point",  # Single-point crossover
    mutation_type="random",         # Random mutation
    mutation_percent_genes=1,  # Mutate 10^-4% of the genes
    on_generation=lambda ga: print(f"Generation: {ga.generations_completed}, Fitness: {ga.best_solution()[1]}")
)

# Run the genetic algorithm
start_time = time.time()
ga_instance.run()
final_time = (time.time() -start_time)/60

print("Execution time (min): " + str(final_time))

# Best solution
solution, solution_fitness, _ = ga_instance.best_solution()
print(f"Selected wavelengths: {np.where(solution == 1)[0]}")
print(f"Best fitness value: {solution_fitness}")

# Plot the fitness values over generations
ga_instance.plot_fitness()

I0000 00:00:1760034370.558309 3057170 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22164 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:06:00.0, compute capability: 8.9
I0000 00:00:1760034371.849488 3057242 service.cc:148] XLA service 0x797dac00a2f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1760034371.849513 3057242 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-10-09 20:26:11.864900: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1760034371.926581 3057242 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1760034372.549257 3057242 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


KeyboardInterrupt: 

In [ ]:



def NRMSEPiqr(observed_values, predicted_values):
    # Calculate RMSEP
    rmsep = np.sqrt(np.mean((observed_values - predicted_values) ** 2))
    # Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = np.percentile(observed_values, 25)
    Q3 = np.percentile(observed_values, 75)

    # Calculate IQR
    IQR = Q3 - Q1

    return rmsep/IQR


NRMSEPiqrscorer = make_scorer(NRMSEPiqr)


trials = 100
random_cv = np.random.seed(777)
seeds = np.random.choice(trials + 1, size=trials, replace= False)
ripartition = 0.25
final_results_pls = dict()


result_NRMSEP_CNN = dict()
NRMSE = []
total_time = 0

epsilon=0.3178
C = 5164
alpha = 0.00122457013067159
for i in range(trials):
    X_train, X_test, Y_train, Y_test = train_test_split(X[:, np.where(solution == 1)[0]], y, test_size=ripartition, random_state=seeds[i])
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    model = build_nn(input_size=X_train_scaled.shape[1])

    early_stopping = EarlyStopping(
            monitor='val_loss',
            min_delta=1e-4,       
            mode='auto',         
            patience=50,
            restore_best_weights=True, 
    )
    start_time = time.time()
    model.fit(X_train_scaled, Y_train, epochs=500, batch_size=32, verbose=0, validation_split=0.2, callbacks=[early_stopping])
    elapsed_time = time.time() - start_time
        
    Y_pred_nn = model.predict(X_test_scaled, verbose=0).flatten()
    NRMSE.append(NRMSEPiqr(Y_test, Y_pred_nn))
    total_time += elapsed_time
    del model
    


result_NRMSEP_CNN["Test size(%)"] =  str(ripartition *100 )
result_NRMSEP_CNN["Mean NRMSEPiqr"] = np.mean(NRMSE)
result_NRMSEP_CNN["Var NRMSEPiqr"] = np.var(NRMSE)
result_NRMSEP_CNN["Mean Training Time (sec)"] = total_time/trials
       


In [ ]:
vip_np = np.array(vip_bands)
vip_np[np.where(solution == 1)[0]]

In [ ]:
waves = np.array(vip_bands)
waves = waves[np.where(solution == 1)[0]]

with open(f"GA_VIP350_CNN_Bands.txt","w") as f:
    f.write("Mean NRMSEPiqr:"+str(result_NRMSEP_CNN["Mean NRMSEPiqr"]) + 
            " , "+ "Var NRMSEPiqr: "+str(result_NRMSEP_CNN["Var NRMSEPiqr"]) + " , " + 
            "Mean Training Time (sec): "+str(result_NRMSEP_CNN["Mean Training Time (sec)"]))
    f.write("\n")
    for wave in waves:
        f.write(str(wave)+"\n")